[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/filippolonghi/medleydb-mert-instrument-classification/blob/main/notebooks/04_segment_and_training_protocol_ablation.ipynb)


# 04 - Segment Length, Data Protocol, Imbalance, and Seed Ablation

## Research question

How sensitive is the selected default configuration to segment duration, subset protocol, class imbalance handling, and random seed?

## Approach

Dataset YAML files build alternative subset CSVs. Experiment YAML files then extract or reuse matching MERT caches and run the same default MLP head. Cache reuse is valid only when data, split, model, layer, pooling, and segment duration match.

## What is fixed

Unless a selected config says otherwise: MedleyDB instrument labels, the `largest_balanced` split, MERT-v1-95M, 5 s segments, validation-based model selection, and test metrics saved only after training.

## What is varied

2 s, 5 s, 10 s segments; subset_40_per_class; capped_natural plain, inverse-frequency loss, weighted sampler; seeds 42, 123, and 2026.

## Expected interpretation

Protocol changes should be reported as robustness checks, not as hidden model-selection sweeps, unless the report states that the protocol itself was selected on validation results.


## What you can change

- `PROJECT_ROOT`, `RUN_ROOT`, and `MEDLEYDB_ROOT` for local vs Colab/Drive paths.
- `EXPERIMENT_GROUPS`, `DATASET_GROUPS`, `MIXTURE_GROUPS`, and `SELECTED_EXPERIMENTS` to run one config at a time.
- `REPLACE_EXISTING = True` only when intentionally overwriting a completed run.


In [ ]:
from pathlib import Path
import os
import shlex
import subprocess
import sys
import yaml
import pandas as pd

PROJECT_ROOT = Path(os.environ.get("PROJECT_ROOT", "/content/medleydb-mert-instrument-classification"))
RUN_ROOT = Path(os.environ.get("RUN_ROOT", "/content/drive/MyDrive/medleydb_mert_project/isolated_stem_v1"))
MEDLEYDB_ROOT = Path(os.environ.get("MEDLEYDB_ROOT", "/content/drive/MyDrive/medleydb_mert_project/MedleyDB"))

os.environ["PROJECT_ROOT"] = str(PROJECT_ROOT)
os.environ["RUN_ROOT"] = str(RUN_ROOT)
os.environ["MEDLEYDB_ROOT"] = str(MEDLEYDB_ROOT)

SUBSET_PROFILE = "largest_balanced"
LABEL_GRANULARITY = "medleydb_instrument"
RESULTS_DIR = RUN_ROOT / "results"
CHECKPOINTS_DIR = RUN_ROOT / "checkpoints"
METADATA_DIR = RUN_ROOT / "data" / "metadata"
CACHE_ROOT = RUN_ROOT / "data" / "cache"
REPORTS_DIR = RUN_ROOT / "data" / "reports"
for path in [RESULTS_DIR, CHECKPOINTS_DIR, METADATA_DIR, CACHE_ROOT, REPORTS_DIR, RUN_ROOT / "configs"]:
    path.mkdir(parents=True, exist_ok=True)

if PROJECT_ROOT.exists():
    os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)
print("Run root:", RUN_ROOT)
print("MedleyDB root:", MEDLEYDB_ROOT)
print("Subset profile:", SUBSET_PROFILE)
print("Label granularity:", LABEL_GRANULARITY)


In [ ]:
def q(path):
    return shlex.quote(str(path))


def run(cmd, check=True):
    print("$", cmd)
    return subprocess.run(cmd, shell=True, check=check)


def load_config(config_path):
    return yaml.safe_load(Path(config_path).read_text(encoding="utf-8"))


def run_experiment_config(config_path, *, extract_embeddings=True, replace_existing=False):
    config_path = Path(config_path)
    cfg = load_config(config_path)
    approach = cfg.get("approach", "frozen_embeddings")
    if approach == "frozen_embeddings" and extract_embeddings:
        run(f"python -m src.features.extract_mert_embeddings --experiment-config {q(config_path)} --batch-size 1 --device auto")
    elif approach == "polyphonic_multilabel" and extract_embeddings:
        run(f"python -m src.features.extract_mert_mixture_embeddings --experiment-config {q(config_path)} --batch-size 1 --device auto")
    args = f"--config {q(config_path)}"
    if replace_existing:
        args += " --replace-existing"
    return run(f"python -m src.experiments.run_experiment {args}")


def run_selected(configs, *, extract_embeddings=True, replace_existing=False):
    for config in configs:
        run_experiment_config(config, extract_embeddings=extract_embeddings, replace_existing=replace_existing)


def build_dataset_config(config_path):
    run(f"python -m src.data.create_balanced_subset --config {q(config_path)}")


## Rebuild and Cache Rules

The 2 s, 10 s, subset_40_per_class, and capped_natural experiments require building a new subset CSV and extracting a separate MERT cache. The 5 s default and seed repeats reuse the official `largest_balanced/medleydb_instrument/layer_last_pool_mean` cache.

## Subset Similarity Diagnostic

`largest_balanced`, `subset_40_per_class`, and `capped_natural` are intended to answer different protocol questions. `largest_balanced` enforces equal or near-equal class counts at the largest feasible per-class size. `subset_40_per_class` uses the same balanced logic but caps each class at 40 examples. `capped_natural` guarantees minimum coverage per class, then fills a global budget without forcing equal class counts. The diagnostic below only reads existing CSVs; it does not rebuild subsets, extract embeddings, modify caches, or affect experiment results.

In [ ]:
import hashlib

SUBSET_DIAGNOSTIC_PATHS = {
    "largest_balanced": RUN_ROOT / "data" / "metadata" / "subset_largest_balanced_medleydb_instrument.csv",
    "subset_40_per_class": RUN_ROOT / "data" / "metadata" / "ablations" / "subset_40_per_class" / "subset_40_per_class_medleydb_instrument.csv",
    "capped_natural": RUN_ROOT / "data" / "metadata" / "ablations" / "capped_natural" / "subset_capped_natural_medleydb_instrument.csv",
}


def subset_fingerprint(frame):
    columns = [column for column in ["segment_id", "track_id", "audio_path", "start_seconds", "duration_seconds", "label_name", "label_id", "split"] if column in frame.columns]
    canonical = frame[columns].sort_values(columns).to_csv(index=False, lineterminator="
")
    return hashlib.sha256(canonical.encode("utf-8")).hexdigest()[:16]


def subset_summary(name, path):
    if not path.exists():
        print(f"{name}: missing ({path})")
        return None
    frame = pd.read_csv(path)
    label_column = "label_name" if "label_name" in frame.columns else "coarse_label"
    counts = frame[label_column].value_counts().sort_index()
    print(f"
{name}: {path}")
    print(f"  rows={len(frame)} classes={counts.size} fingerprint={subset_fingerprint(frame)}")
    print(f"  min/max class count={int(counts.min())}/{int(counts.max())}")
    display(counts.rename("count").reset_index().rename(columns={"index": "label"}))
    return frame

loaded_subsets = {name: subset_summary(name, path) for name, path in SUBSET_DIAGNOSTIC_PATHS.items()}

print("
Pairwise segment overlap")
names = [name for name, frame in loaded_subsets.items() if frame is not None]
for i, left in enumerate(names):
    for right in names[i + 1:]:
        left_ids = set(loaded_subsets[left]["segment_id"].astype(str))
        right_ids = set(loaded_subsets[right]["segment_id"].astype(str))
        intersection = len(left_ids & right_ids)
        union = len(left_ids | right_ids)
        print(f"  {left} vs {right}: shared={intersection} union={union} jaccard={intersection / union if union else 0:.3f}")

if len(names) >= 2:
    hashes = {name: subset_fingerprint(loaded_subsets[name]) for name in names}
    duplicates = [(a, b) for i, a in enumerate(names) for b in names[i + 1:] if hashes[a] == hashes[b]]
    if duplicates:
        print("Warning: identical subset fingerprints detected:", duplicates)
    else:
        print("No identical subset fingerprints detected among available CSVs.")


In [ ]:
DATASET_GROUPS = {
    "segment_length": [
        "configs/datasets/subset_largest_balanced_medleydb_instrument_2s.yaml",
        "configs/datasets/subset_largest_balanced_medleydb_instrument_5s.yaml",
        "configs/datasets/subset_largest_balanced_medleydb_instrument_10s.yaml",
    ],
    "protocol": [
        "configs/datasets/subset_40_per_class_medleydb_instrument.yaml",
        "configs/datasets/subset_capped_natural_medleydb_instrument.yaml",
    ],
}
EXPERIMENT_GROUPS = {
    "segment_length": [
        "configs/ablations/protocol_segment_2s_largest_balanced_medleydb_mert95_last_mean_mlp_h256_d02.yaml",
        "configs/ablations/protocol_segment_5s_largest_balanced_medleydb_mert95_last_mean_mlp_h256_d02.yaml",
        "configs/ablations/protocol_segment_10s_largest_balanced_medleydb_mert95_last_mean_mlp_h256_d02.yaml",
    ],
    "imbalance_protocol": [
        "configs/ablations/protocol_subset_40_per_class_medleydb_mert95_last_mean_mlp_h256_d02.yaml",
        "configs/ablations/protocol_capped_natural_plain_medleydb_mert95_last_mean_mlp_h256_d02.yaml",
        "configs/ablations/protocol_capped_natural_inverse_loss_medleydb_mert95_last_mean_mlp_h256_d02.yaml",
        "configs/ablations/protocol_capped_natural_weighted_sampler_medleydb_mert95_last_mean_mlp_h256_d02.yaml",
    ],
    "seed_repeats": [
        "configs/ablations/protocol_seed42_largest_balanced_medleydb_mert95_last_mean_mlp_h256_d02.yaml",
        "configs/ablations/protocol_seed123_largest_balanced_medleydb_mert95_last_mean_mlp_h256_d02.yaml",
        "configs/ablations/protocol_seed2026_largest_balanced_medleydb_mert95_last_mean_mlp_h256_d02.yaml",
    ],
}
SELECTED_DATASETS = []
SELECTED_EXPERIMENTS = [
    "configs/ablations/protocol_seed42_largest_balanced_medleydb_mert95_last_mean_mlp_h256_d02.yaml",
]
REPLACE_EXISTING = False

for dataset_config in SELECTED_DATASETS:
    build_dataset_config(dataset_config)
run_selected(SELECTED_EXPERIMENTS, extract_embeddings=True, replace_existing=REPLACE_EXISTING)
